# 模块 & 包管理 · 知识点

### 1、模块是什么

In [ ]:
# 一个 .py 文件就是一个"模块"（module），文件名（去掉 .py）就是模块名
# 比如有个文件 math_utils.py，里面写：
#   def add(a, b):
#       return a + b
# 别的文件就可以 import math_utils，然后用 math_utils.add(1, 2) 调用

# 模块存在的意义：把代码拆到多个文件里，按功能组织，而不是所有代码堆在一个文件里
# 对比 Java：一个 .py 模块大致对应 Java 里的一个类文件，但 Python 模块里不一定只放一个类，
# 可以直接放函数、变量，没有"一个文件必须对应一个类"的强制要求

### 2、用 %%writefile 创建一个真实模块并导入

In [ ]:
%%writefile math_utils.py
# %%writefile 是 Jupyter 的魔法命令，会把这个 cell 剩下的内容写成一个真实的 .py 文件
# 运行这个 cell 后，当前目录下会出现 math_utils.py 文件

def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

PI = 3.14159

In [ ]:
# 现在 math_utils.py 是一个真实存在的文件，可以正常 import 了
import math_utils

print(math_utils.add(1, 2))
print(math_utils.PI)

# from 模块 import 具体的名字，用的时候不用写模块名前缀
from math_utils import multiply
print(multiply(3, 4))

# import 模块 as 别名，给模块取个短名字，常见于第三方库（比如 import numpy as np）
import math_utils as mu
print(mu.add(5, 6))

### 3、`if __name__ == "__main__":` 的作用

In [ ]:
%%writefile greet.py
def greet(name):
    print(f"Hello, {name}")

# __name__ 是模块的一个内置变量：
# - 如果这个文件是被直接运行的（python greet.py），__name__ 的值是 "__main__"
# - 如果这个文件是被别的文件 import 的，__name__ 的值是模块名 "greet"
# 用这个判断可以实现"既能被导入复用，又能单独运行测试"
if __name__ == "__main__":
    greet("直接运行测试")

In [ ]:
# 被 import 的时候，if __name__ == "__main__": 里的代码不会执行
import greet
greet.greet("Tom")    # 只会打印这一行，"直接运行测试"那行不会出现

# 用命令行直接跑这个文件，才会触发 if __name__ == "__main__": 里的代码
# !python greet.py    # 会打印 "Hello, 直接运行测试"

# 这个写法在 Java 里没有直接对应物，有点像每个文件都可以有自己的 main 方法，
# 但只有"被当作启动入口跑"的时候才会执行

### 4、模块搜索路径 sys.path

In [ ]:
# import 一个模块时，Python 按 sys.path 里的目录顺序去找同名的 .py 文件/包
import sys
print(sys.path)   # 一个目录列表，包含：当前脚本所在目录、标准库目录、第三方库安装目录(site-packages)等

# 这就是为什么 math_utils.py 放在当前目录就能直接 import：
# 当前目录默认在 sys.path 里

# 想临时让 Python 也去别的目录找模块，可以手动加进去（不推荐常用，了解即可）
# sys.path.append("/some/other/dir")

### 5、包（package）：目录 + `__init__.py`

In [ ]:
# 模块是单个 .py 文件，包是一个"目录"，目录里有一个 __init__.py 文件就会被当成包
# __init__.py 可以是空文件，它的存在本身就是"这是一个包"的标记（3.3+ 其实没有它也能算包，但写了更明确、兼容性更好)

import os
os.makedirs("mypkg", exist_ok=True)

In [ ]:
%%writefile mypkg/__init__.py
# 包的初始化文件，import mypkg 时会先执行这个文件
# 可以在这里做一些初始化逻辑，或者把子模块的东西"提升"到包的顶层，方便外部使用
print("mypkg 被导入了")

In [ ]:
%%writefile mypkg/helper.py
def say_hi():
    print("来自 mypkg.helper 的问候")

In [ ]:
# 导入包里的子模块，用点号
import mypkg.helper
mypkg.helper.say_hi()

# 或者只导入子模块里的具体函数
from mypkg.helper import say_hi
say_hi()

### 6、`__all__` 控制 `from xxx import *`

In [ ]:
%%writefile mymodule.py
def public_func():
    pass

def _private_func():    # 单下划线开头，约定内部使用
    pass

__all__ = ["public_func"]   # 显式声明 from mymodule import * 时只导出这些名字

In [ ]:
from mymodule import *
public_func()        # 能用
# _private_func()    # 报错 NameError，因为 __all__ 没包含它，* 导入不会带上它

# 没写 __all__ 的话，from xxx import * 默认会导入所有不以下划线开头的名字
# 实际项目里很少用 import *（容易污染命名空间、不知道具体导入了什么），这里了解机制即可

### 7、pip 包管理

In [ ]:
# pip 是 Python 的包管理工具，对应 Java 的 Maven/Gradle 下载依赖
# 常用命令（在终端运行，不是在 notebook 里）：

# pip install requests          # 安装一个第三方包
# pip install requests==2.31.0  # 安装指定版本
# pip uninstall requests        # 卸载
# pip list                      # 列出当前环境已安装的包
# pip show requests             # 查看某个包的详细信息（版本、依赖等）

# pip freeze > requirements.txt   # 把当前环境所有包和版本号导出到文件，对应 Java 的 pom.xml/build.gradle 依赖声明
# pip install -r requirements.txt # 根据 requirements.txt 安装所有依赖，新机器/新同事一键还原环境

# 用 ! 前缀可以在 notebook 里直接跑终端命令
!pip --version

### 8、虚拟环境 venv

In [ ]:
# 虚拟环境：每个项目独立一套 Python 包，避免不同项目之间的依赖版本互相冲突
# 对应 Java：相当于每个项目有自己独立的本地依赖隔离，不会因为全局装了某个库的不同版本而互相打架

# 创建虚拟环境（在项目根目录下）：
# python -m venv .venv

# 激活虚拟环境：
# macOS/Linux: source .venv/bin/activate
# Windows:     .venv\Scripts\activate
# 激活后命令行提示符前面会出现 (.venv)，此时 pip install 装的包只在这个虚拟环境里生效

# 退出虚拟环境：
# deactivate

# .venv 目录通常会加进 .gitignore，不提交到代码仓库（本项目根目录下就有这个配置）

### 9、典型项目结构

In [ ]:
# 一个常见的 Python 项目结构（对应 Java Maven 的 src/main/java 标准目录）：
#
# myproject/
# ├── .venv/                # 虚拟环境，不提交 git
# ├── src/
# │   └── myproject/        # 真正的包，里面是模块和子包
# │       ├── __init__.py
# │       ├── main.py
# │       └── utils.py
# ├── tests/                 # 测试代码
# │   └── test_utils.py
# ├── requirements.txt       # 依赖声明，对应 pom.xml
# ├── .gitignore
# └── README.md
#
# 对比 Java/Maven：
# - src/myproject/   大致对应 src/main/java/com/xxx/
# - tests/           大致对应 src/test/java/
# - requirements.txt 大致对应 pom.xml 里的 <dependencies>
# 区别：Python 项目结构没有强制标准，团队/框架不同会有差异，这里是最常见的一种写法